The core idea of an RNN: process words one at a time, in order, and carry a "memory" from each word to the next. The so-called memory is called a hidden state, and it gets updated after every token, and to get the next hidden state we use the previous one + the next token. In the end we have a final sentiment for our text, represented by the final hidden layer.

In [1]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 53.9 MB/s eta 0:00:00


In [2]:
import re
import torch
from torch import nn, optim
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api

In [3]:
# Get the word embeddings from gensim
glove = api.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [4]:
# Use GPU instead of CPU for faster results
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
dataset = load_dataset('imdb')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
# Function to clean up text and return it word by word (token)
def tokenize(text):
  text = re.sub(r'<.*?>', ' ', text)  # remove HTML tags
  text = text.lower()
  text = text.replace("'", " '").replace("!", " !").replace("?", " ?")
  return text.split()

In [7]:
class Vocabulary():
  def __init__(self):
    self.wordmap = {'<pad>' : 0,
                    '<unk>' : 1}

  def build(self, sentences):
    for sentence in sentences:
      sentence = tokenize(sentence)
      for word in sentence:
        if word not in self.wordmap.keys():
          self.wordmap[word] = max(self.wordmap.values()) + 1

    return None

  def to_indices(self, text):
    indices = []
    for word in tokenize(text):
      if word not in self.wordmap.keys():
        indices.append(1)
      else:
        indices.append(self.wordmap[word])

    return indices

In [8]:
# Get the 10k most frequent words for the vocbulary
def extract_common_words(limit=10000):
  reviews = dataset['train']['text']
  word_counts = dict()

  for review in reviews:
    review = tokenize(review)
    for word in review:
      if word not in word_counts.keys():
        word_counts[word] = 1
      else:
        word_counts[word] += 1

  sorted_words = sorted(word_counts, key=word_counts.get, reverse=True)
  most_common_words = sorted_words[:limit]

  return most_common_words

In [9]:
# Create our vocabulary with the 10k most common words in IMDB reviews
words = extract_common_words()
vocab = Vocabulary()
vocab.build(words)

In [10]:
# In L17 I used a model that averages out the words from a sentence
# and pays no attention to word order, an issue that RNNs address.
class RNNSentimentModel(nn.Module): # Inherit from nn.Module
    # Define the architecture of the model
    def __init__(self):
      super().__init__()
      # Create the embedding table (10000 rows, 1 for each word, each with 50 dimensions)
      self.embedding = nn.Embedding(
                          num_embeddings=len(vocab.wordmap),
                          embedding_dim=50
                      )

      # Create the RNN that takes a 50D token vector and has a hidden state of size 128
      self.rnn = nn.RNN(input_size=50, hidden_size=128, batch_first=True)
      self.linear = nn.Linear(128, 1)

    # Define the process of the model
    def forward(self, x):
        x = self.embedding(x)
        out, hidden = self.rnn(x) # out is every hidden state, hidden is the very last one
        hidden = hidden.squeeze(0) # shape (1, batch, 128) -> (batch, 128)
        x = torch.sigmoid(self.linear(hidden)) # feed the last 128D hidden state to the neural network and apply sigmoid

        return x

In [12]:
def get_review_indices(review):
  indices = vocab.to_indices(review)
  if len(indices) > 256:
    indices = indices[:256]
  else:
    indices += [0] * (256 - len(indices))  # 0 = <pad> index
  return indices

In [13]:
# Prepare data for training/testing
review_tensors_train = []
labels_train = dataset['train']['label']

review_tensors_test = []
labels_test = dataset['test']['label']

for review in dataset['train']['text']:
  review_tensors_train.append(torch.tensor(get_review_indices(review), dtype=torch.long))

for review in dataset['test']['text']:
  review_tensors_test.append(torch.tensor(get_review_indices(review), dtype=torch.long))

X_train = torch.stack(review_tensors_train)
y_train = torch.tensor(labels_train, dtype=torch.float32).reshape(-1, 1)

X_test = torch.stack(review_tensors_test)
y_test = torch.tensor(labels_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [14]:
# Create the model
model = RNNSentimentModel()
model = model.to(device)

In [15]:
# Set the embedding tables to existing words in our table
# To gensim's values for a better optimized start
for word, idx in vocab.wordmap.items():
    if word in glove:
        model.embedding.weight.data[idx] = torch.tensor(glove[word])

In [16]:
# Define the loss function and the optimizer
loss_ft = nn.BCELoss()
opt = optim.Adam(model.parameters(), lr=0.01)

In [17]:
n_epochs = 40

for epoch in range(n_epochs):

  batch_loss = 0

  # Train
  model.train()
  for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    pred = model(X_batch)
    loss = loss_ft(pred, y_batch)
    loss.backward()
    opt.step()
    opt.zero_grad()

    batch_loss += loss.item()

  avg_batch_loss = batch_loss / len(train_loader)

  if epoch % (n_epochs / 10) == 0 or epoch == n_epochs - 1:
    print(f'Epoch {epoch}')
    print(f'Loss: {avg_batch_loss:.4f}')

Epoch 0
Loss: 0.7094
Epoch 4
Loss: 0.6953
Epoch 8
Loss: 0.6848
Epoch 12
Loss: 0.6777
Epoch 16
Loss: 0.6669
Epoch 20
Loss: 0.6560
Epoch 24
Loss: 0.6566
Epoch 28
Loss: 0.6557
Epoch 32
Loss: 0.6542
Epoch 36
Loss: 0.6564
Epoch 39
Loss: 0.6543


In [19]:
model.eval()
total_correct = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        preds = (model(X_batch) > 0.5).float()
        total_correct += (preds == y_batch).sum().item()
print(f"Test accuracy: {total_correct / len(test_dataset):.4f}")

Test accuracy: 0.5062


We only get ~50% accuracy although RNNs were supposed to improve our averaging model. This is because RNN's are best used for shorter texts, since they have a 'short-term memory' due to the vanishing gradient problem. Basically our model completely forgets older words, hence our predictions cant be accurate, and the results are no better than random guessing.